<a href="https://colab.research.google.com/github/tfxhk/urdu_ocr_codesaviours_si26_Hafiza_Tehreem/blob/main/SI26_Week3_Tehreem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SI26 – Week 3 – Hafiza Tehreem Fatima

## Urdu OCR Dataset Expansion and PyTorch Dataset Preparation

This notebook expands the Urdu OCR dataset to over 200 images, updates the dataset for training, and builds a custom PyTorch Dataset class to prepare the data for OCR model training using Microsoft's TrOCR processor.

# Install library

In [ ]:
!pip install transformers torch pillow pandas opencv-python-headless

# import libraries

In [2]:
import os
import glob
import zipfile
import cv2
import torch
import pandas as pd

from PIL import Image
from torch.utils.data import Dataset
from transformers import TrOCRProcessor

# extract dataset

In [4]:
zip_path = "/content/Urdu OCR zip file new.zip"
extract_path = "/content/data/raw"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    for file in zip_ref.namelist():
        try:
            zip_ref.extract(file, extract_path)
        except Exception:
            pass

print("Dataset extracted successfully!")

Dataset extracted successfully!


# count images

In [5]:
import os
from collections import Counter

image_extensions = ('.png', '.jpg', '.jpeg', '.jfif', '.bmp', '.tif', '.tiff', '.webp')

counter = Counter()
total = 0

for root, dirs, files in os.walk("/content/data/raw"):
    for file in files:
        ext = os.path.splitext(file)[1].lower()
        if ext in image_extensions:
            counter[ext] += 1
            total += 1

print("Image count by extension:")
for ext, count in counter.items():
    print(f"{ext}: {count}")

print("\nTotal images:", total)

Image count by extension:
.jfif: 87
.png: 287
.webp: 1

Total images: 375


# update labels.csv

In [6]:
import os
import pandas as pd

dataset_root = "/content/data/raw/Urdu OCR"

image_extensions = (".png", ".jpg", ".jpeg", ".jfif", ".webp")

rows = []

for root, dirs, files in os.walk(dataset_root):
    for file in files:
        if file.lower().endswith(image_extensions):
            rows.append({
                "image": os.path.join(root, file),
                "text": ""
            })

labels = pd.DataFrame(rows)

labels = labels.sort_values("image").reset_index(drop=True)

os.makedirs("/content/data", exist_ok=True)

labels.to_csv(
    "/content/data/labels.csv",
    index=False,
    encoding="utf-8-sig"
)

print("labels.csv created successfully!")
print("Total images:", len(labels))

labels.head()

labels.csv created successfully!
Total images: 375


,image,text
0,/content/data/raw/Urdu OCR/Characters/1.PNG,
1,/content/data/raw/Urdu OCR/Characters/10.PNG,
2,/content/data/raw/Urdu OCR/Characters/100.PNG,
3,/content/data/raw/Urdu OCR/Characters/101.PNG,
4,/content/data/raw/Urdu OCR/Characters/102.PNG,


In [7]:
import os
import pandas as pd

dataset_root = "/content/data/raw/Urdu OCR"

image_extensions = (".png", ".jpg", ".jpeg", ".jfif", ".webp")

rows = []

for root, dirs, files in os.walk(dataset_root):
    for file in files:
        if file.lower().endswith(image_extensions):
            rows.append({
                "image": os.path.join(root, file),
                "text": "Urdu Text"
            })

df = pd.DataFrame(rows)

os.makedirs("/content/data", exist_ok=True)

df.to_csv(
    "/content/data/labels.csv",
    index=False,
    encoding="utf-8-sig"
)

print("CSV Created")
print("Total Images:", len(df))

CSV Created
Total Images: 375


# Load TrOCR Processor

In [8]:
from transformers import TrOCRProcessor

processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-printed")
print("Processor loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Processor loaded successfully!


In [9]:
processor = TrOCRProcessor.from_pretrained(
    "microsoft/trocr-base-printed"
)

# Build Dataset Class

In [10]:
import torch
import pandas as pd

from PIL import Image

from torch.utils.data import Dataset, DataLoader

from transformers import TrOCRProcessor

In [11]:
class UrduOCRDataset(Dataset):

    def __init__(self, csv_path, processor):
        self.data = pd.read_csv(csv_path)
        self.processor = processor
        print(f"Dataset loaded: {len(self.data)} samples")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        row = self.data.iloc[idx]

        image = Image.open(row["image"]).convert("RGB")

        encoding = self.processor(
            image,
            return_tensors="pt"
        )

        pixel_values = encoding.pixel_values.squeeze()

        labels = self.processor.tokenizer(
            row["text"],
            padding="max_length",
            max_length=128,
            truncation=True
        ).input_ids

        labels = torch.tensor(labels)

        return {
            "pixel_values": pixel_values,
            "labels": labels
        }

# create dataset

In [12]:
dataset = UrduOCRDataset(
    "/content/data/labels.csv",
    processor
)

print("Dataset Size:", len(dataset))

Dataset loaded: 375 samples
Dataset Size: 375


# test one sample

In [13]:
sample = dataset[0]

print("Pixel Values Shape:", sample["pixel_values"].shape)
print("Labels Shape:", sample["labels"].shape)

Pixel Values Shape: torch.Size([3, 384, 384])
Labels Shape: torch.Size([128])


# Create Train/Test Split

In [14]:
from torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size]
)

print("Training Samples:", train_size)
print("Testing Samples:", test_size)

Training Samples: 300
Testing Samples: 75


In [15]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

print("Train batches:", len(train_loader))
print("Test batches:", len(test_loader))
print("DataLoaders are ready for Week 4!")

Train batches: 38
Test batches: 10
DataLoaders are ready for Week 4!


In Week 3, I expanded my Urdu OCR dataset to more than 200 images increasing it to 375 images collected from multiple sources
I updated the dataset structure, generated the labels.csv file & reused the preprocessing pipeline from Week 2
I built a custom PyTorch Dataset class to load image-label pairs and process them using the Microsoft TrOCR Processor
Finally, I tested the dataset successfully, created an 80/20 training-testing split & prepared DataLoaders for model training

# Week 4

In [16]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


# Load Model

In [17]:
import transformers
import tokenizers

print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)

Transformers: 4.41.2
Tokenizers: 0.19.1


In [19]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

print(device)

processor = TrOCRProcessor.from_pretrained(
    "microsoft/trocr-base-printed"
)

model = VisionEncoderDecoderModel.from_pretrained(
    "microsoft/trocr-base-printed"
)

model.to(device)

model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

print("Model Loaded Successfully")

cuda


Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-printed and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model Loaded Successfully


In [20]:
from transformers import TrOCRProcessor

processor = TrOCRProcessor.from_pretrained(
    "microsoft/trocr-base-printed"
)

print("Processor loaded successfully!")

Processor loaded successfully!


# Training Setup

In [21]:
from transformers import VisionEncoderDecoderModel
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model = VisionEncoderDecoderModel.from_pretrained(
    "microsoft/trocr-base-printed"
)

model = model.to(device)

model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

print("Model loaded successfully!")

Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-printed and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded successfully!


In [22]:
from torch.utils.data import DataLoader
from transformers import AdamW

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4)

optimizer = AdamW(model.parameters(), lr=5e-5)

print("Training batches:", len(train_loader))
print("Ready to train!")

/usr/local/lib/python3.12/dist-packages/transformers/optimization.py:588: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Training batches: 75
Ready to train!


In [23]:
print(dataset)

In [24]:
print("dataset:", "dataset" in globals())
print("train_dataset:", "train_dataset" in globals())
print("test_dataset:", "test_dataset" in globals())
print("processor:", "processor" in globals())

dataset: True
train_dataset: True
test_dataset: True
processor: True


# Load the TrOCR model

In [28]:
from transformers import VisionEncoderDecoderModel
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model = VisionEncoderDecoderModel.from_pretrained(
    "microsoft/trocr-base-printed"
)

model = model.to(device)

model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

print("Model loaded successfully!")

Using device: cuda


Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-printed and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded successfully!


# Create DataLoaders

In [26]:
from torch.utils.data import DataLoader
from transformers import AdamW

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4)

optimizer = AdamW(model.parameters(), lr=5e-5)

print("Training batches:", len(train_loader))
print("Ready to train!")

Training batches: 75
Ready to train!


/usr/local/lib/python3.12/dist-packages/transformers/optimization.py:588: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


# Train the TrOCR Model

In [29]:
num_epochs = 3

for epoch in range(num_epochs):

    model.train()
    total_loss = 0

    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-"*30)

    for batch_idx, batch in enumerate(train_loader):

        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            pixel_values=pixel_values,
            labels=labels
        )

        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 10 == 0:
            print(
                f"Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}"
            )

    avg_loss = total_loss / len(train_loader)

    print(f"Epoch {epoch+1} Complete | Average Loss: {avg_loss:.4f}")

print("\nTraining Complete!")


Epoch 1/3
------------------------------
Batch 0/75 | Loss: 17.8492
Batch 10/75 | Loss: 18.6233
Batch 20/75 | Loss: 18.2966
Batch 30/75 | Loss: 19.2203
Batch 40/75 | Loss: 17.9777
Batch 50/75 | Loss: 18.0384
Batch 60/75 | Loss: 18.4905
Batch 70/75 | Loss: 19.0427
Epoch 1 Complete | Average Loss: 18.6689

Epoch 2/3
------------------------------
Batch 0/75 | Loss: 19.2024
Batch 10/75 | Loss: 19.0143
Batch 20/75 | Loss: 19.0830
Batch 30/75 | Loss: 18.5056
Batch 40/75 | Loss: 18.0567
Batch 50/75 | Loss: 18.7079
Batch 60/75 | Loss: 19.1188
Batch 70/75 | Loss: 19.2511
Epoch 2 Complete | Average Loss: 18.6417

Epoch 3/3
------------------------------
Batch 0/75 | Loss: 19.3257
Batch 10/75 | Loss: 18.5520
Batch 20/75 | Loss: 18.6050
Batch 30/75 | Loss: 18.9262
Batch 40/75 | Loss: 18.7700
Batch 50/75 | Loss: 18.4327
Batch 60/75 | Loss: 18.8296
Batch 70/75 | Loss: 18.6732
Epoch 3 Complete | Average Loss: 18.6613

Training Complete!


# Evaluate the Model

In [31]:
model.eval()

print("Model Evaluation on Test Images")
print()

correct = 0
total = 0

with torch.no_grad():

    for batch in test_loader:

        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"]

        generated_ids = model.generate(pixel_values)

        generated_text = processor.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        actual_text = processor.batch_decode(
            labels,
            skip_special_tokens=True
        )

        for pred, actual in zip(generated_text, actual_text):

            total += 1

            if pred.strip() == actual.strip():
                correct += 1

            print("Predicted:", pred)
            print("Actual:", actual)
            print()

accuracy = (correct / total) * 100 if total > 0 else 0

print(f"Accuracy: {accuracy:.1f}% ({correct}/{total} correct)")

Model Evaluation on Test Images

Predicted: TAX
Actual: Urdu Text

Predicted: IL
Actual: Urdu Text

Predicted: &
Actual: Urdu Text

Predicted: :
Actual: Urdu Text

Predicted: L
Actual: Urdu Text

Predicted: 5
Actual: Urdu Text

Predicted: U
Actual: Urdu Text

Predicted: 1
Actual: Urdu Text

Predicted: SR:
Actual: Urdu Text

Predicted: 1
Actual: Urdu Text

Predicted: 2
Actual: Urdu Text

Predicted: 0
Actual: Urdu Text

Predicted: PAX
Actual: Urdu Text

Predicted: 2
Actual: Urdu Text

Predicted: AMOUNT NO. :
Actual: Urdu Text

Predicted: ***
Actual: Urdu Text

Predicted: S
Actual: Urdu Text

Predicted: U
Actual: Urdu Text

Predicted: U
Actual: Urdu Text

Predicted: RAX
Actual: Urdu Text

Predicted: IT
Actual: Urdu Text

Predicted: LSP
Actual: Urdu Text

Predicted: 1
Actual: Urdu Text

Predicted: P
Actual: Urdu Text

Predicted: U
Actual: Urdu Text

Predicted: ENTE PERM CREW AE & (PK
Actual: Urdu Text

Predicted: :
Actual: Urdu Text

Predicted: TAX
Actual: Urdu Text

Predicted: CASHIER
Act

In [32]:
!apt-get update -qq
!apt-get install -y tesseract-ocr
!apt-get install -y tesseract-ocr-urd

!pip install pytesseract pandas pillow tqdm

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 162 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  tesseract-ocr-urd
0 upgraded, 1 newly installed, 0 to remove and 162 not upgraded.
Need to get 1,000 kB of archives.
After this operation, 1,413 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-urd all 1:4.00~git30-7274cfa-1.1 [1,000 kB]
Fetched 1,000 kB in 2s (486 kB/s)
Selecting previously unselected package tesseract-ocr-urd.
(Reading database ... 122403 files and dire

In [37]:
import pytesseract
from PIL import Image
import pandas as pd
import os
import glob
from tqdm import tqdm

In [38]:
pytesseract.pytesseract.tesseract_cmd = "/usr/bin/tesseract"

In [39]:
dataset_path = "/content/data/raw/Urdu OCR"

extensions = [
    "*.png",
    "*.jpg",
    "*.jpeg",
    "*.jfif",
    "*.webp"
]

all_images = []

for ext in extensions:
    all_images.extend(
        glob.glob(
            os.path.join(dataset_path, "**", ext),
            recursive=True
        )
    )

print("Total Images Found:", len(all_images))

Total Images Found: 150


In [40]:
labels = []

for image_path in tqdm(all_images):

    try:

        img = Image.open(image_path)

        text = pytesseract.image_to_string(
            img,
            lang="urd"
        ).strip()

        labels.append({
            "image": image_path,
            "text": text
        })

    except Exception as e:

        print("Skipped:", image_path)

labels_df = pd.DataFrame(labels)

labels_df.to_csv(
    "/content/data/labels_auto.csv",
    index=False,
    encoding="utf-8"
)

print("Done!")
print("Labels Generated:", len(labels_df))

100%|██████████| 150/150 [03:30<00:00,  1.40s/it]

Done!
Labels Generated: 150


In [41]:
labels = pd.read_csv("/content/data/labels_auto.csv")

labels.head(20)

,image,text
0,/content/data/raw/Urdu OCR/hand written/textbo...,أْ اب ے‫ مک لہ ے\n‎٠ ٌ 1‏ ا ََ ھ2 تج - - حم
1,/content/data/raw/Urdu OCR/hand written/textbo...,NaN
2,/content/data/raw/Urdu OCR/hand written/textbo...,7 7 ر 2ا رامرے ہگے۔
3,/content/data/raw/Urdu OCR/hand written/textbo...,ضر ۷ن سو ٹا ہے لٴہ\nے‌ ل ز4ا ۔ ےانھو ںپ۔ہے 4 سے
4,/content/data/raw/Urdu OCR/hand written/textbo...,۱ مم 7\nہے یل ب نا
5,/content/data/raw/Urdu OCR/hand written/textbo...,30 ڑ ۱\n۰ت ل کا زرییرع مبىیا ‎٤‏\n‏کے سد
6,/content/data/raw/Urdu OCR/hand written/textbo...,NaN
7,/content/data/raw/Urdu OCR/hand written/textbo...,NaN
8,/content/data/raw/Urdu OCR/hand written/textbo...,سم\n\nلم ٘\nبل کر\n2\nکَٔ\nگروں\nدو\n-۰۔دے\nکل...
9,/content/data/raw/Urdu OCR/hand written/textbo...,NaN


In [43]:
import shutil

shutil.copy(
    "/content/data/labels_auto.csv",
    "/content/data/labels.csv"
)

print("labels.csv Updated Successfully!")

labels.csv Updated Successfully!


In [45]:
class UrduOCRDataset(Dataset):

    def __init__(self, csv_path, processor):
        self.data = pd.read_csv(csv_path)
        self.processor = processor
        print(f"Dataset loaded: {len(self.data)} samples")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        row = self.data.iloc[idx]

        image = Image.open(row["image"]).convert("RGB")

        encoding = self.processor(
            image,
            return_tensors="pt"
        )

        pixel_values = encoding.pixel_values.squeeze()

        labels = self.processor.tokenizer(
            row["text"],
            padding="max_length",
            max_length=128,
            truncation=True
        ).input_ids

        labels = torch.tensor(labels)

        return {
            "pixel_values": pixel_values,
            "labels": labels
        }

In [46]:
dataset = UrduOCRDataset(
    "/content/data/labels.csv",
    processor
)

print("Dataset Size:", len(dataset))

Dataset loaded: 150 samples
Dataset Size: 150


In [47]:
from torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size]
)

print("Training Samples:", train_size)
print("Testing Samples:", test_size)

Training Samples: 120
Testing Samples: 30


In [48]:
from transformers import VisionEncoderDecoderModel
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model = VisionEncoderDecoderModel.from_pretrained(
    "microsoft/trocr-base-printed"
)

model = model.to(device)

model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

print("Model loaded successfully!")

Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-printed and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded successfully!


In [49]:
num_epochs = 3

for epoch in range(num_epochs):

    model.train()
    total_loss = 0

    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-"*30)

    for batch_idx, batch in enumerate(train_loader):

        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            pixel_values=pixel_values,
            labels=labels
        )

        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 10 == 0:
            print(
                f"Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}"
            )

    avg_loss = total_loss / len(train_loader)

    print(f"Epoch {epoch+1} Complete | Average Loss: {avg_loss:.4f}")

print("\nTraining Complete!")


Epoch 1/3
------------------------------
Batch 0/75 | Loss: 17.4272
Batch 10/75 | Loss: 18.5703
Batch 20/75 | Loss: 19.3743
Batch 30/75 | Loss: 18.0338
Batch 40/75 | Loss: 18.2811
Batch 50/75 | Loss: 18.8512
Batch 60/75 | Loss: 18.1239
Batch 70/75 | Loss: 17.8867
Epoch 1 Complete | Average Loss: 18.6543

Epoch 2/3
------------------------------
Batch 0/75 | Loss: 19.6056
Batch 10/75 | Loss: 19.1153
Batch 20/75 | Loss: 18.2149
Batch 30/75 | Loss: 18.6037
Batch 40/75 | Loss: 18.7368
Batch 50/75 | Loss: 19.1471
Batch 60/75 | Loss: 18.9209
Batch 70/75 | Loss: 17.7703
Epoch 2 Complete | Average Loss: 18.6713

Epoch 3/3
------------------------------
Batch 0/75 | Loss: 18.0753
Batch 10/75 | Loss: 18.4989
Batch 20/75 | Loss: 18.8512
Batch 30/75 | Loss: 18.4849
Batch 40/75 | Loss: 18.0806
Batch 50/75 | Loss: 19.3844
Batch 60/75 | Loss: 18.6690
Batch 70/75 | Loss: 18.9490
Epoch 3 Complete | Average Loss: 18.6608

Training Complete!


In [50]:
model.eval()

print("Model Evaluation on Test Images")
print()

correct = 0
total = 0

with torch.no_grad():

    for batch in test_loader:

        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"]

        generated_ids = model.generate(pixel_values)

        generated_text = processor.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        actual_text = processor.batch_decode(
            labels,
            skip_special_tokens=True
        )

        for pred, actual in zip(generated_text, actual_text):

            total += 1

            if pred.strip() == actual.strip():
                correct += 1

            print("Predicted:", pred)
            print("Actual:", actual)
            print()

accuracy = (correct / total) * 100 if total > 0 else 0

print(f"Accuracy: {accuracy:.1f}% ({correct}/{total} correct)")

Model Evaluation on Test Images



/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1168: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Predicted: TAX
Actual: Urdu Text

Predicted: IL
Actual: Urdu Text

Predicted: &
Actual: Urdu Text

Predicted: :
Actual: Urdu Text

Predicted: L
Actual: Urdu Text

Predicted: 5
Actual: Urdu Text

Predicted: U
Actual: Urdu Text

Predicted: 1
Actual: Urdu Text

Predicted: SR:
Actual: Urdu Text

Predicted: 1
Actual: Urdu Text

Predicted: 2
Actual: Urdu Text

Predicted: 0
Actual: Urdu Text

Predicted: PAX
Actual: Urdu Text

Predicted: 2
Actual: Urdu Text

Predicted: AMOUNT NO. :
Actual: Urdu Text

Predicted: ***
Actual: Urdu Text

Predicted: S
Actual: Urdu Text

Predicted: U
Actual: Urdu Text

Predicted: U
Actual: Urdu Text

Predicted: RAX
Actual: Urdu Text

Predicted: IT
Actual: Urdu Text

Predicted: LSP
Actual: Urdu Text

Predicted: 1
Actual: Urdu Text

Predicted: P
Actual: Urdu Text

Predicted: U
Actual: Urdu Text

Predicted: ENTE PERM CREW AE & (PK
Actual: Urdu Text

Predicted: :
Actual: Urdu Text

Predicted: TAX
Actual: Urdu Text

Predicted: CASHIER
Actual: Urdu Text

Predicted: 1
Actu